# Distress Flag and Features

### Load Libraries

In [1]:
import pandas as pd

### Load data

In [2]:
ipi_wide_full = pd.read_csv("data/ipi_wide.csv", parse_dates = ["date"], dtype = {"division": str})

## Distress Flag Definition

In [3]:
N = 3 # 3 months in a row

# negative growth_yoy
# for each row, 1 if YOY is negative and 0 if not
negative_yoy = (ipi_wide_full["growth_yoy"] < 0).astype(float).where(ipi_wide_full["growth_yoy"].notna())

# how many of the last N months were negative?
# for each sector separately, add the last N values of negative_yoy
rolled = negative_yoy.groupby(ipi_wide_full["division"]).transform(lambda s: s.rolling(N).sum())

# * Distress Flag
# if the sum equals N (all of the last N months were negative), otherwise 0
# blank where there aren't N months of YOY yet
ipi_wide_full["distress"] = (rolled == N).astype(float).where(rolled.notna())

# Check counts
print(ipi_wide_full["distress"].value_counts(dropna=False))
print(ipi_wide_full["distress"].mean())

# Check that blanks are only at the start of each sector
ipi_wide_full[ipi_wide_full["distress"].isna()].groupby("division")["date"].agg(["min", "max", "count"])

# Check a sector
x = ipi_wide_full[ipi_wide_full["division"] == "22"]   # Rubber & plastics
print(x.loc[x["distress"] == 1, ["date", "growth_yoy", "distress"]].head(10))

distress
0.0    2767
NaN     336
1.0     281
Name: count, dtype: int64
0.09219160104986876
           date  growth_yoy  distress
1775 2021-12-01      -5.198       1.0
1776 2022-01-01     -10.178       1.0
1777 2022-02-01     -11.018       1.0
1778 2022-03-01     -15.957       1.0
1779 2022-04-01     -10.990       1.0
1780 2022-05-01     -14.968       1.0
1781 2022-06-01     -14.395       1.0
1782 2022-07-01      -8.617       1.0
1783 2022-08-01      -4.222       1.0
1784 2022-09-01      -2.987       1.0


### Heatmap

## Target Label Definition

In [4]:
# * target: flag 6 months ahead within each sector
# Is the sector still in distress 6 months later?
ipi_wide_full["target"] = ipi_wide_full.groupby("division")["distress"].shift(-6)  # groupby prevents boundary leaks
# shift(6): 6 months backwards
# shift(-6): 6 months ahead

## Feature Definition

In [5]:
# * YOY lagged by 1 (shifted up 1)
ipi_wide_full["yoy_lag_1"] = ipi_wide_full.groupby("division")["growth_yoy"].shift(1)

# * YOY lagged by 3 (shifted up 3)
ipi_wide_full["yoy_lag_3"] = ipi_wide_full.groupby("division")["growth_yoy"].shift(3)

# * 6 month rolling average (growth_yoy)
# .rolling(6) : sum the last 6 months
# .mean() : mean of last 6 months
ipi_wide_full["yoy_roll_6"] = ipi_wide_full.groupby("division")["growth_yoy"].transform(lambda s: s.rolling(6).mean())

# * 6 month volatility of MoM
ipi_wide_full["mom_vol_6"] = ipi_wide_full.groupby("division")["growth_mom"].transform(lambda s: s.rolling(6).std())

## Save ipi_wide_full as csv

In [6]:
ipi_wide_full.to_csv("data/ipi_wide_full.csv", index = False)